In [1]:
import os
import pandas as pd
import numpy as np
import s3fs
from sqlalchemy import create_engine
from dotenv import load_dotenv

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import joblib
import json

load_dotenv()

False

In [2]:
S3_BUCKET_CURATED = os.getenv("S3_BUCKET_CURATED", "curated")
S3_BUCKET_RAW = os.getenv("S3_BUCKET_RAW", "raw")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin123")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio:9000")
MINIO_SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB = os.getenv("POSTGRES_DB", "oil_pipeline")
POSTGRES_USER = os.getenv("POSTGRES_USER", "oil_user")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "oil_password")

def get_s3_fs():
    protocol = "https" if MINIO_SECURE else "http"
    return s3fs.S3FileSystem(
        key=MINIO_ACCESS_KEY,
        secret=MINIO_SECRET_KEY,
        client_kwargs={"endpoint_url": f"{protocol}://{MINIO_ENDPOINT}"},
    )

def get_pg_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
        f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
    )

In [3]:
fs = get_s3_fs()

telemetry = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/clean/well_telemetry/well_telemetry_clean.parquet",
    filesystem=fs
)

engine = get_pg_engine()
well_targets = pd.read_sql("SELECT * FROM well_targets", engine)

print(f"Telemetry: {telemetry.shape}")
print(f"Well targets: {well_targets.shape}")
telemetry.head(3)

Telemetry: (2136, 11)
Well targets: (90, 4)


,record_id,well_id,timestamp,pump_speed_rpm,pump_current,pressure_in,pressure_out,temperature,vibration,oil_flow_rate,date
0,1,1,2025-10-01 00:00:00,1470.0,58.2,95.3,122.4,88.1,1.4,8.8,2025-10-01
1,2,1,2025-10-01 01:00:00,1468.0,58.5,95.1,122.1,88.2,1.5,8.9,2025-10-01
2,3,1,2025-10-01 02:00:00,1472.0,58.0,94.9,121.8,88.3,1.6,8.7,2025-10-01


In [4]:
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"])
telemetry["date"] = telemetry["timestamp"].dt.date
telemetry["date"] = pd.to_datetime(telemetry["date"])

telemetry_daily = (
    telemetry.groupby(["well_id", "date"], as_index=False)
    .agg(
        avg_pressure_in=("pressure_in", "mean"),
        avg_pressure_out=("pressure_out", "mean"),
        avg_temperature=("temperature", "mean"),
        avg_pump_current=("pump_current", "mean"),
        avg_pump_speed_rpm=("pump_speed_rpm", "mean"),
        avg_vibration=("vibration", "mean"),
        avg_oil_flow_rate=("oil_flow_rate", "mean"),
        max_vibration=("vibration", "max"),
        std_pressure_out=("pressure_out", "std"),
    )
)

print(f"Telemetry daily: {telemetry_daily.shape}")
telemetry_daily.head(3)

Telemetry daily: (89, 11)


,well_id,date,avg_pressure_in,avg_pressure_out,avg_temperature,avg_pump_current,avg_pump_speed_rpm,avg_vibration,avg_oil_flow_rate,max_vibration,std_pressure_out
0,1,2025-10-01,95.070833,122.233333,88.287500,58.395833,1472.541667,1.462500,8.808333,1.60,0.291423
1,1,2025-10-02,94.993333,122.055000,87.755417,58.207083,1469.530000,1.796667,8.810833,2.00,0.321897
2,1,2025-10-03,94.948750,121.991667,88.030000,58.115000,1469.327500,1.725000,8.797083,1.99,0.350375


In [5]:
well_targets["date"] = pd.to_datetime(well_targets["date"])

dataset = telemetry_daily.merge(
    well_targets[["well_id", "date", "daily_oil_ton"]],
    on=["well_id", "date"],
    how="inner"
)

print(f"Dataset shape: {dataset.shape}")
print(f"NULL values:\n{dataset.isnull().sum()}")
dataset.head(5)

Dataset shape: (89, 12)
NULL values:
well_id               0
date                  0
avg_pressure_in       0
avg_pressure_out      0
avg_temperature       0
avg_pump_current      0
avg_pump_speed_rpm    0
avg_vibration         0
avg_oil_flow_rate     0
max_vibration         0
std_pressure_out      0
daily_oil_ton         0
dtype: int64


,well_id,date,avg_pressure_in,avg_pressure_out,avg_temperature,avg_pump_current,avg_pump_speed_rpm,avg_vibration,avg_oil_flow_rate,max_vibration,std_pressure_out,daily_oil_ton
0,1,2025-10-01,95.070833,122.233333,88.287500,58.395833,1472.541667,1.462500,8.808333,1.60,0.291423,212.4
1,1,2025-10-02,94.993333,122.055000,87.755417,58.207083,1469.530000,1.796667,8.810833,2.00,0.321897,213.8
2,1,2025-10-03,94.948750,121.991667,88.030000,58.115000,1469.327500,1.725000,8.797083,1.99,0.350375,211.9
3,1,2025-10-04,94.972500,122.054583,87.948750,58.188333,1469.740833,1.791667,8.815833,1.98,0.294338,215.1
4,1,2025-10-05,94.914583,122.057083,88.155833,58.277500,1469.266667,1.785000,8.743333,2.00,0.286819,214.6


In [6]:
feature_cols = [
    "avg_pressure_in",
    "avg_pressure_out",
    "avg_temperature",
    "avg_pump_current",
    "avg_pump_speed_rpm",
    "avg_vibration",
    "avg_oil_flow_rate",
    "max_vibration",
    "std_pressure_out",
]

target_col = "daily_oil_ton"

dataset_clean = dataset.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

X = dataset_clean[feature_cols]
y = dataset_clean[target_col]

print(f"Dataset after cleaning: {dataset_clean.shape}")
print(f"Features: {feature_cols}")
print(f"Target: {target_col}")

Dataset after cleaning: (89, 12)
Features: ['avg_pressure_in', 'avg_pressure_out', 'avg_temperature', 'avg_pump_current', 'avg_pump_speed_rpm', 'avg_vibration', 'avg_oil_flow_rate', 'max_vibration', 'std_pressure_out']
Target: daily_oil_ton


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

Train size: 71
Test size: 18


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f"Linear Regression:")
print(f"  MAE:  {mae_lr:.4f}")
print(f"  RMSE: {rmse_lr:.4f}")

Linear Regression:
  MAE:  0.9834
  RMSE: 1.2254


In [10]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"Random Forest:")
print(f"  MAE:  {mae_rf:.4f}")
print(f"  RMSE: {rmse_rf:.4f}")

Random Forest:
  MAE:  0.8685
  RMSE: 1.0655


In [11]:
metrics_df = pd.DataFrame({
    "model": ["LinearRegression", "RandomForest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf]
})

metrics_df

,model,MAE,RMSE
0,LinearRegression,0.983433,1.225424
1,RandomForest,0.868515,1.065490


In [12]:
os.makedirs("/home/jovyan/work/models", exist_ok=True)

joblib.dump(lr_model, "/home/jovyan/work/models/lr_rate_model.pkl")
joblib.dump(rf_model, "/home/jovyan/work/models/rf_rate_model.pkl")
joblib.dump(scaler, "/home/jovyan/work/models/scaler.pkl")

metrics = {
    "linear_regression": {"MAE": mae_lr, "RMSE": rmse_lr},
    "random_forest": {"MAE": mae_rf, "RMSE": rmse_rf}
}

with open("/home/jovyan/work/models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Models and metrics saved.")

Models and metrics saved.


In [13]:
best_model = rf_model if mae_rf < mae_lr else lr_model
best_preds = y_pred_rf if mae_rf < mae_lr else y_pred_lr
best_name = "RandomForest" if mae_rf < mae_lr else "LinearRegression"

print(f"Best model: {best_name}")

actual_vs_predicted = dataset_clean.iloc[X_test.index].copy()
actual_vs_predicted = actual_vs_predicted[["well_id", "date", "daily_oil_ton"]].copy()
actual_vs_predicted["predicted_oil_ton"] = best_preds
actual_vs_predicted["error"] = (
    actual_vs_predicted["predicted_oil_ton"] - actual_vs_predicted["daily_oil_ton"]
)
actual_vs_predicted["abs_error"] = actual_vs_predicted["error"].abs()
actual_vs_predicted["model"] = best_name

actual_vs_predicted = actual_vs_predicted.sort_values("date").reset_index(drop=True)
actual_vs_predicted.head(10)

Best model: RandomForest


,well_id,date,daily_oil_ton,predicted_oil_ton,error,abs_error,model
0,2,2025-10-01,185.9,185.544792,-0.355208,0.355208,RandomForest
1,1,2025-10-01,212.4,213.413293,1.013293,1.013293,RandomForest
2,5,2025-10-03,199.3,198.333118,-0.966882,0.966882,RandomForest
3,2,2025-10-04,187.3,185.438322,-1.861678,1.861678,RandomForest
4,1,2025-10-05,214.6,213.617521,-0.982479,0.982479,RandomForest
5,5,2025-10-06,199.1,198.432252,-0.667748,0.667748,RandomForest
6,2,2025-10-10,185.9,185.626677,-0.273323,0.273323,RandomForest
7,1,2025-10-11,211.4,213.677427,2.277427,2.277427,RandomForest
8,1,2025-10-13,214.9,213.030358,-1.869642,1.869642,RandomForest
9,2,2025-10-13,186.4,185.620035,-0.779965,0.779965,RandomForest


In [14]:
all_preds_rf = rf_model.predict(dataset_clean[feature_cols])
all_preds_lr = lr_model.predict(scaler.transform(dataset_clean[feature_cols]))

full_forecast = dataset_clean[["well_id", "date", "daily_oil_ton"]].copy()
full_forecast["predicted_rf"] = all_preds_rf
full_forecast["predicted_lr"] = all_preds_lr
full_forecast["error_rf"] = full_forecast["predicted_rf"] - full_forecast["daily_oil_ton"]
full_forecast["error_lr"] = full_forecast["predicted_lr"] - full_forecast["daily_oil_ton"]
full_forecast = full_forecast.sort_values(["well_id", "date"]).reset_index(drop=True)

full_forecast.head(10)

,well_id,date,daily_oil_ton,predicted_rf,predicted_lr,error_rf,error_lr
0,1,2025-10-01,212.4,213.413293,214.720940,1.013293,2.320940
1,1,2025-10-02,213.8,213.772312,213.432311,-0.027688,-0.367689
2,1,2025-10-03,211.9,212.233988,212.752462,0.333988,0.852462
3,1,2025-10-04,215.1,214.294439,213.467799,-0.805561,-1.632201
4,1,2025-10-05,214.6,213.617521,213.187141,-0.982479,-1.412859
5,1,2025-10-06,213.2,213.414161,212.636036,0.214161,-0.563964
6,1,2025-10-07,211.7,212.576294,213.372606,0.876294,1.672606
7,1,2025-10-08,212.5,212.549860,213.187709,0.049860,0.687709
8,1,2025-10-09,213.9,213.233831,213.019775,-0.666169,-0.880225
9,1,2025-10-10,212.0,212.590616,213.252371,0.590616,1.252371


In [15]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
1,avg_pressure_out,0.205609
0,avg_pressure_in,0.183371
2,avg_temperature,0.176179
6,avg_oil_flow_rate,0.151606
4,avg_pump_speed_rpm,0.114260
3,avg_pump_current,0.107020
7,max_vibration,0.036920
5,avg_vibration,0.024637
8,std_pressure_out,0.000398


In [16]:
engine = get_pg_engine()

actual_vs_predicted.to_sql(
    "mart_actual_vs_predicted",
    engine,
    if_exists="replace",
    index=False
)

full_forecast.to_sql(
    "mart_full_forecast",
    engine,
    if_exists="replace",
    index=False
)

metrics_df.to_sql(
    "mart_model_metrics",
    engine,
    if_exists="replace",
    index=False
)

feature_importance.to_sql(
    "mart_feature_importance",
    engine,
    if_exists="replace",
    index=False
)

print("All ML marts saved to PostgreSQL.")

All ML marts saved to PostgreSQL.


In [17]:
print("=" * 50)
print("ML Rate Forecast — Summary")
print("=" * 50)
print(f"Dataset size: {dataset_clean.shape[0]} rows")
print(f"Train/Test split: 80/20")
print()
print(f"Linear Regression:  MAE={mae_lr:.3f}, RMSE={rmse_lr:.3f}")
print(f"Random Forest:      MAE={mae_rf:.3f}, RMSE={rmse_rf:.3f}")
print()
print(f"Best model: {best_name}")
print()
print("Top-3 important features:")
print(feature_importance.head(3).to_string(index=False))

ML Rate Forecast — Summary
Dataset size: 89 rows
Train/Test split: 80/20

Linear Regression:  MAE=0.983, RMSE=1.225
Random Forest:      MAE=0.869, RMSE=1.065

Best model: RandomForest

Top-3 important features:
         feature  importance
avg_pressure_out    0.205609
 avg_pressure_in    0.183371
 avg_temperature    0.176179
